![Ph](https://t3.ftcdn.net/jpg/05/72/86/02/360_F_572860227_WZYTIQSRSBCA4CYdOtvGE4x3J6pJVGEo.jpg)

# 📘 Project Guide

## Student Performance — BI Analysis with Python

**Goal:** Build a portfolio-ready educational BI analysis from `StudentsPerformance.csv`.

### How to use this notebook
1. Put `StudentsPerformance.csv` in the same folder as this notebook.
2. Install the required packages if needed:
   ```bash
   pip install pandas numpy matplotlib seaborn scipy jupyter
   ```
3. Open the notebook in Jupyter Notebook, JupyterLab, or VS Code.
4. Run the cells **from top to bottom** the first time.
5. After that, individual analysis sections can be rerun independently.

### Notebook structure
- **Section 1:** Setup & data loading
- **Section 2:** Data Quality
- **Section 3:** KPI & EDA
- **Section 4:** Segment Intelligence
- **Section 5:** Statistical / Educational Analysis
- **Section 6:** Data Modeling
- **Section 7:** Executive Dashboard
- **Section 8:** Executive Summary
- **Section 9:** Portfolio / CV
- **Section 10:** Limitations

### Analysis rules
- No invented time trends: the source has **no date/time column**.
- `race/ethnicity` is treated only as the dataset's category code.
- Gaps are associations, not automatic causal explanations.
- The original CSV is read-only; all derived fields are created in Python objects.


# Student Performance — BI Analysis in Python

**Portfolio-grade Educational Data Analysis**

This notebook analyzes the Kaggle **Students Performance in Exams** dataset using Python.

### Objectives
- Data quality audit
- Exploratory performance analysis
- Segment intelligence instead of time intelligence
- Gap and root-cause breakdowns
- Statistical significance testing
- Correlation analysis
- Executive dashboard-style visualizations
- Star-schema recommendation
- Executive summary and ATS-friendly CV bullets

> **Important:** The dataset contains no date/time field, so MoM/QoQ/YoY trend analysis is not applicable. No time trend is fabricated.


# 1️⃣ Setup & Data Loading

**Purpose:** Import libraries, load the source CSV, and create analysis-friendly column names.

**Expected output:** A preview of the source data.

**Important:** Do not modify the raw CSV manually. This notebook keeps the source unchanged.


In [ ]:
# Imports and display setup
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid", context="notebook")

DATA_PATH = Path("StudentsPerformance.csv")
if not DATA_PATH.exists():
    # Fallback for the supplied environment
    DATA_PATH = Path("/kaggle/input/datasets/spscientist/students-performance-in-exams/StudentsPerformance.csv")

df = pd.read_csv(DATA_PATH)

# Standardize column names for analysis while preserving the source values.
df = df.rename(columns={
    "race/ethnicity": "race_ethnicity",
    "parental level of education": "parental_education",
    "test preparation course": "test_prep",
    "math score": "math_score",
    "reading score": "reading_score",
    "writing score": "writing_score",
})

score_cols = ["math_score", "reading_score", "writing_score"]
cat_cols = ["gender", "race_ethnicity", "parental_education", "lunch", "test_prep"]

df.head()


## 1. Data Quality Report

# 2️⃣ Data Quality — Missing Values & Duplicates

**Question:** Is the source clean enough for analysis?

Check:
- number of rows/columns
- missing cells
- exact duplicate rows

**Interpretation:** Zero missing/duplicate records means no basic imputation or deduplication is required.


In [ ]:
# Basic structure
quality = pd.DataFrame({
    "rows": [len(df)],
    "columns": [df.shape[1]],
    "duplicate_rows": [df.duplicated().sum()],
    "missing_cells": [df.isna().sum().sum()]
})
quality


# 2️⃣ Data Quality — Score Validation

**Question:** Are exam scores valid?

Valid range: **0–100**.

The table below checks minimum, maximum, and the number of values outside that range for each subject.


In [ ]:
# Missing values
missing = df.isna().sum().sort_values(ascending=False).to_frame("missing_count")
missing["missing_pct"] = missing["missing_count"] / len(df) * 100
missing


# 2️⃣ Data Quality — Category Consistency

**Question:** Are categorical labels consistent?

We inspect:
- unique labels
- leading/trailing spaces
- case-insensitive duplicate labels

This prevents silent segmentation errors.


In [ ]:
# Score range validation
range_check = pd.DataFrame({
    "column": score_cols,
    "min": [df[c].min() for c in score_cols],
    "max": [df[c].max() for c in score_cols],
    "outside_0_100": [((df[c] < 0) | (df[c] > 100)).sum() for c in score_cols]
})
range_check


# 2️⃣ Data Quality — Logical Consistency

The dataset is a **single flat table** and contains no joins, so cross-table logical consistency cannot be tested.

We can still validate score types and known categorical values.


In [ ]:
# Category-label consistency checks
category_report = []
for c in cat_cols:
    vals = sorted(df[c].dropna().astype(str).unique())
    category_report.append({
        "column": c,
        "unique_values": len(vals),
        "values": " | ".join(vals),
        "leading_trailing_space_values": sum(v != v.strip() for v in vals),
        "lowercase_duplicates": len(vals) - len({v.strip().lower() for v in vals}),
    })
category_report = pd.DataFrame(category_report)
category_report


# 2️⃣ Data Quality — Cleaning Decision

If the checks above return zero issues, the correct BI decision is **not to clean data unnecessarily**.

Recommended practice:
- preserve raw source
- document validation
- standardize names only in the analysis layer
- keep a data-quality log


In [ ]:
# Logical consistency checks
# This is a single flat table with no joins or repeated student records.
# Therefore, cross-table join inconsistencies cannot occur in the supplied source.
logical_checks = {
    "rows_with_non_numeric_scores": int(
        df[score_cols].apply(pd.to_numeric, errors="coerce").isna().any(axis=1).sum()
    ),
    "rows_with_impossible_test_prep_value": int((~df["test_prep"].isin(["completed", "none"])).sum()),
    "rows_with_impossible_lunch_value": int((~df["lunch"].isin(["standard", "free/reduced"])).sum()),
}
pd.Series(logical_checks, name="count")


# 3️⃣ Performance Analysis — KPIs

**Business questions:**
- What is the average score by subject?
- What percentage passes all subjects?
- How spread out are overall scores?

**Pass definition used here:** a student passes when **Math, Reading, and Writing are all ≥ 60**.


### Cleaning recommendation

The supplied file is already clean enough for analysis: no missing cells, exact duplicate rows, out-of-range scores, or inconsistent category labels were detected. The only transformation applied in this notebook is **renaming columns for Python readability**; the source data itself is not overwritten.

Because there is no join key or second source, cross-table logical consistency checks are not applicable.


# 3️⃣ Performance Analysis — Subject Correlations

**Question:** How strongly do Math, Reading, and Writing move together?

Pearson correlation is used for the continuous score variables.

A high correlation means scores tend to move together; it does **not** prove that one subject causes another.


## 2. Performance Analysis — EDA

# 3️⃣ Performance Analysis — Segment Comparison

Compare average performance across:
- gender
- race/ethnicity
- parental education
- lunch
- test preparation

Always look at both the **mean score and sample size (n)** before interpreting a segment.


In [ ]:
# Derived measures
df["overall_score"] = df[score_cols].mean(axis=1)
df["all_subjects_pass"] = (df[score_cols] >= 60).all(axis=1)

overall_pass_rate = df["all_subjects_pass"].mean() * 100

kpis = pd.DataFrame({
    "Metric": [
        "Students", "Math average", "Reading average", "Writing average",
        "Overall average", "All-subject pass rate", "Overall score std dev"
    ],
    "Value": [
        len(df),
        df["math_score"].mean(),
        df["reading_score"].mean(),
        df["writing_score"].mean(),
        df["overall_score"].mean(),
        overall_pass_rate,
        df["overall_score"].std()
    ]
})
kpis


# 3️⃣ Performance Analysis — Visual EDA

Charts make segment differences easier to communicate.

**Portfolio tip:** When presenting this project, explain the business question first, then the chart, then the numerical evidence.


In [ ]:
# Correlations among subjects
corr = df[score_cols].corr(method="pearson")
corr


# 3️⃣ Performance Analysis — Top & Bottom 10%

Define:
- Top 10% = overall score at or above the 90th percentile
- Bottom 10% = overall score at or below the 10th percentile

Use these groups to understand what distinguishes very high and very low performance profiles.


In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="Blues", vmin=0, vmax=1)
plt.title("Correlation Between Subject Scores")
plt.tight_layout()
plt.show()


# 4️⃣ Segment Intelligence — Core Gaps

Because there is **no date field**, classic Time Intelligence (MoM/QoQ/YoY) does not apply.

We replace it with **Segment Intelligence**:
- Test prep completed vs none
- Standard vs free/reduced lunch
- Parental education groups


In [ ]:
# Segment summaries
def segment_summary(col):
    out = df.groupby(col)[score_cols + ["overall_score", "all_subjects_pass"]].agg(
        {**{c: "mean" for c in score_cols + ["overall_score"]},
         "all_subjects_pass": "mean"}
    )
    out["pass_rate_pct"] = out.pop("all_subjects_pass") * 100
    out["n"] = df.groupby(col).size()
    return out.sort_values("overall_score", ascending=False)

for col in ["gender", "race_ethnicity", "parental_education", "lunch", "test_prep"]:
    print(f"\n### {col}")
    display(segment_summary(col).round(2))


# 4️⃣ Segment Intelligence — Parental Education

Rank parental education categories by overall score.

Then identify the largest pairwise gap.

**Rule:** A large descriptive gap is a signal for investigation, not proof of a causal mechanism.


In [ ]:
# Visual comparison across key segments
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.ravel(), ["gender", "lunch", "test_prep", "race_ethnicity"]):
    tmp = df.groupby(col)[score_cols].mean().reset_index().melt(
        id_vars=col, var_name="subject", value_name="average"
    )
    sns.barplot(data=tmp, x=col, y="average", hue="subject", ax=ax)
    ax.set_title(f"Average Scores by {col}")
    ax.set_ylim(0, 100)
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


# 4️⃣ Root Cause Breakdown — Lunch Gap

**Question:** Does the lunch-type gap remain after controlling descriptively for parental education?

If the gap persists within multiple education groups, parental education alone does not explain the observed difference.

This is still an observational association.


In [ ]:
# Top/bottom 10% performer profiles
q90 = df["overall_score"].quantile(0.90)
q10 = df["overall_score"].quantile(0.10)

top10 = df[df["overall_score"] >= q90]
bottom10 = df[df["overall_score"] <= q10]

profile = pd.DataFrame({
    "group": ["Top 10%", "Bottom 10%"],
    "n": [len(top10), len(bottom10)],
    "overall_avg": [top10["overall_score"].mean(), bottom10["overall_score"].mean()],
    "math_avg": [top10["math_score"].mean(), bottom10["math_score"].mean()],
    "reading_avg": [top10["reading_score"].mean(), bottom10["reading_score"].mean()],
    "writing_avg": [top10["writing_score"].mean(), bottom10["writing_score"].mean()],
    "pass_rate_pct": [top10["all_subjects_pass"].mean()*100, bottom10["all_subjects_pass"].mean()*100],
})
profile.round(2)


# 4️⃣ Root Cause Breakdown — Largest Education Gap

Break the largest parental-education gap down by subject.

**Goal:** identify which subject contributes most to the overall point difference.


## 3. Segment Intelligence — Gap Analysis & Root Causes

There is no date dimension in the source, so segment comparisons replace time intelligence.

# 4️⃣ Root Cause Breakdown — Test Prep by Gender

Check whether the test-prep association is similar for males and females.

This helps determine whether the overall test-prep gap is broadly consistent or concentrated in a subgroup.


In [ ]:
# Helper for two-group gap analysis
def two_group_gap(col, group_a, group_b):
    a = df[df[col] == group_a][score_cols + ["overall_score"]].mean()
    b = df[df[col] == group_b][score_cols + ["overall_score"]].mean()
    gap = a - b
    pct = gap / b * 100
    return pd.DataFrame({"group_a": a, "group_b": b, "gap_points": gap, "gap_pct_vs_b": pct}).round(3)

prep_gap = two_group_gap("test_prep", "completed", "none")
lunch_gap = two_group_gap("lunch", "standard", "free/reduced")

print("Test-prep completed vs none")
display(prep_gap)

print("Standard lunch vs free/reduced lunch")
display(lunch_gap)


# 4️⃣ Root Cause Interpretation Rules

### Use these rules in the final presentation
- **Gap:** descriptive difference between groups.
- **Persistence after stratification:** evidence that another measured variable does not fully explain the gap.
- **Cause:** should not be claimed from this observational dataset.
- **Recommendation:** phrase as an intervention to evaluate, not as a guaranteed solution.


In [ ]:
# Parental education ranking and largest pairwise gap
edu = segment_summary("parental_education").reset_index()
edu_rank = edu[["parental_education", "overall_score", "pass_rate_pct", "n"]].sort_values(
    "overall_score", ascending=False
)
display(edu_rank.round(2))

pairwise = []
groups = edu_rank["parental_education"].tolist()
means = dict(zip(edu_rank["parental_education"], edu_rank["overall_score"]))
for i, g1 in enumerate(groups):
    for g2 in groups[i+1:]:
        pairwise.append((g1, g2, means[g1] - means[g2]))
pairwise_df = pd.DataFrame(pairwise, columns=["higher_group", "lower_group", "gap_points"])
display(pairwise_df.loc[pairwise_df["gap_points"].abs().idxmax()].to_frame("largest_education_gap"))


# 5️⃣ Statistical Analysis — Test Preparation

**Question:** Is the test-prep score difference statistically meaningful?

Use Welch's independent-samples t-test because group variances do not need to be assumed equal.

Interpret alongside:
- mean difference
- p-value
- practical magnitude


In [ ]:
# Root-cause breakdown 1: Does the lunch gap persist within parental education?
lunch_by_edu = df.groupby(["parental_education", "lunch"])[score_cols + ["overall_score"]].mean().reset_index()

pivot_lunch_edu = lunch_by_edu.pivot(index="parental_education", columns="lunch", values="overall_score")
pivot_lunch_edu["standard_minus_free_reduced"] = (
    pivot_lunch_edu["standard"] - pivot_lunch_edu["free/reduced"]
)
display(pivot_lunch_edu.round(2))

print(
    "Interpretation: if the lunch gap remains positive across most education groups, "
    "the association is not explained away by parental education alone. "
    "This is evidence of persistence, not proof of causality."
)


# 5️⃣ Statistical Analysis — Gender

Test whether the observed female-vs-male differences are statistically distinguishable from zero for each subject and overall score.

Do not collapse subject-specific patterns into one statement.


In [ ]:
# Root-cause breakdown 2: What subjects drive the largest parental-education gap?
largest_pair = pairwise_df.loc[pairwise_df["gap_points"].abs().idxmax()]
g1, g2 = largest_pair["higher_group"], largest_pair["lower_group"]

subject_gap = (
    df[df["parental_education"].isin([g1, g2])]
    .groupby("parental_education")[score_cols]
    .mean()
    .T
)
subject_gap["gap_points"] = subject_gap[g1] - subject_gap[g2]
subject_gap["gap_pct_vs_lower"] = subject_gap["gap_points"] / subject_gap[g2] * 100
subject_gap.round(2)


# 5️⃣ Statistical Analysis — Parental Education

Two complementary tests:
- **One-way ANOVA:** do category means differ?
- **Spearman correlation:** is there a monotonic relationship when education categories are treated as an ordered scale?

Neither test proves causation.


In [ ]:
# Root-cause breakdown 3: test-prep gap by subject and gender
prep_gender = df.groupby(["gender", "test_prep"])[score_cols + ["overall_score"]].mean().reset_index()
display(prep_gender.round(2))

prep_pivot = prep_gender.pivot(index="gender", columns="test_prep", values="overall_score")
prep_pivot["completed_minus_none"] = prep_pivot["completed"] - prep_pivot["none"]
prep_pivot.round(2)


# 5️⃣ Visualization — Test Prep Gap

Use this chart as an executive communication visual.

**Talking point:** Which subject shows the largest average difference between students who completed preparation and those who did not?


### Root-cause interpretation rules

- A **gap** is descriptive; it does not prove causation.
- A gap that remains after stratifying by another variable is evidence that the second variable does not fully account for the observed difference.
- Recommended actions are framed as **opportunities to test/intervene**, not claims that a variable causes performance.


# 5️⃣ Educational Interpretation

### Key interpretation framework
- Test preparation: association with higher scores; identify the strongest subject gap.
- Gender: subject-specific pattern.
- Parental education: differences across categories plus strength of ordered association.
- Lunch: persistent gap is an equity-monitoring signal.

Avoid statements such as “X causes higher scores” unless the study design supports causal inference.


## 4. Business / Educational Analysis

# 6️⃣ Data Modeling — Star Schema

**Recommended BI model:**

`Fact_StudentScores`
→ `Dim_Gender`
→ `Dim_Ethnicity`
→ `Dim_ParentalEducation`
→ `Dim_Lunch`
→ `Dim_TestPrep`

**Why Star Schema?**
- low-cardinality dimensions
- simpler Power BI relationships
- easier DAX
- easier filtering
- clearer semantic model

A generated `student_id` is only a surrogate key because the source has no real student identifier.


In [ ]:
# Welch t-tests: completed vs none
tests = []
for subject in score_cols + ["overall_score"]:
    completed = df.loc[df["test_prep"] == "completed", subject]
    none = df.loc[df["test_prep"] == "none", subject]
    t, p = stats.ttest_ind(completed, none, equal_var=False)
    tests.append({
        "metric": subject,
        "completed_mean": completed.mean(),
        "none_mean": none.mean(),
        "gap_points": completed.mean() - none.mean(),
        "t_stat": t,
        "p_value": p,
        "significant_at_0.05": p < 0.05
    })
test_prep_stats = pd.DataFrame(tests)
test_prep_stats.round(4)


# 6️⃣ Data Modeling — Python Preview

This cell creates a reproducible preview of the fact table and dimensions.

**Do not interpret `student_id` as an actual student identity.**


In [ ]:
# Gender significance tests
gender_tests = []
for subject in score_cols + ["overall_score"]:
    male = df.loc[df["gender"] == "male", subject]
    female = df.loc[df["gender"] == "female", subject]
    t, p = stats.ttest_ind(female, male, equal_var=False)
    gender_tests.append({
        "metric": subject,
        "female_mean": female.mean(),
        "male_mean": male.mean(),
        "female_minus_male": female.mean() - male.mean(),
        "p_value": p,
        "significant_at_0.05": p < 0.05
    })
gender_stats = pd.DataFrame(gender_tests)
gender_stats.round(4)


# 7️⃣ Executive Dashboard

### Dashboard design guide

The dashboard answers four executive questions:

1. **How are students performing overall?**
2. **Where are the largest segment gaps?**
3. **Which subjects move together?**
4. **What is the most important evidence-backed issue to investigate?**

The largest detected gap is automatically highlighted with a short evidence-based callout.


In [ ]:
# Parental education: one-way ANOVA
edu_groups = [g["overall_score"].values for _, g in df.groupby("parental_education")]
anova_f, anova_p = stats.f_oneway(*edu_groups)

# Spearman correlation between ordered education level and overall score.
edu_order = {
    "high school": 1,
    "some high school": 2,
    "some college": 3,
    "associate's degree": 4,
    "bachelor's degree": 5,
    "master's degree": 6,
}
edu_numeric = df["parental_education"].map(edu_order)
rho, rho_p = stats.spearmanr(edu_numeric, df["overall_score"])

education_stats = pd.DataFrame({
    "test": ["One-way ANOVA", "Spearman rank correlation"],
    "statistic": [anova_f, rho],
    "p_value": [anova_p, rho_p],
})
education_stats


# 8️⃣ Executive Notebook — Presentation Summary

This section is written as the narrative you can use in a portfolio presentation.

Recommended storytelling order:
**Highlights → KPIs → Insights → Evidence/Root Causes → Risks → Opportunities → Recommendations → Next Steps**


In [ ]:
# Subject gap visualization for test preparation
prep_subject = df.groupby("test_prep")[score_cols].mean().T
prep_subject["gap_completed_minus_none"] = prep_subject["completed"] - prep_subject["none"]

ax = prep_subject[["completed", "none"]].plot(kind="bar", figsize=(10, 6))
ax.set_title("Test Preparation: Average Subject Scores")
ax.set_ylabel("Average score")
ax.set_ylim(0, 100)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

prep_subject.round(2)


# 9️⃣ Portfolio / CV Summary

These bullets are written for direct use in a Data/BI Analyst portfolio or CV.

They emphasize:
- scale
- analytical methods
- quantified findings
- BI/data-modeling skills
- business/educational question answered


### Educational interpretation

**Test preparation:** completion is associated with higher scores in all three subjects, with the largest descriptive gain in writing. The Welch tests above assess whether those mean differences are statistically distinguishable from zero.

**Gender:** the analysis separates subject-specific gaps rather than collapsing them into one overall statement.

**Parental education:** the ANOVA tests whether average performance differs across education categories; the Spearman result tests the strength of a monotonic ordered relationship. Neither establishes a causal pathway.

**Equity risk:** persistent gaps by lunch type and parental education are important monitoring signals. They should be treated as areas for targeted support evaluation rather than causal explanations.


# 🔟 Limitations & Responsible Interpretation

Always mention these limitations in a portfolio presentation:

- no time dimension
- no school/geography
- single-source sample
- no external validation of categorical fields
- observational data ≠ causation
- no real student ID
- no intervention/randomization data


## 5. Data Modeling — Recommended Star Schema

### Fact_StudentScores
**Grain:** one row per student.

Columns:
- `student_id` — generated surrogate key
- `gender_key`
- `ethnicity_key`
- `parental_education_key`
- `lunch_key`
- `test_prep_key`
- `math_score`
- `reading_score`
- `writing_score`

### Dimensions
- **Dim_Gender**
- **Dim_Ethnicity**
- **Dim_ParentalEducation**
- **Dim_Lunch**
- **Dim_TestPrep**

A **star schema** is preferable to snowflaking because each dimension is low-cardinality and independent. Keeping dimensions directly connected to the fact table simplifies Power BI relationships, DAX, filtering, and report performance.

The source has no student identifier, so `student_id` should be generated as a surrogate key only; it must not be interpreted as a real-world student ID.


In [ ]:
# Create a reproducible star-schema preview
fact = df.copy()
fact.insert(0, "student_id", np.arange(1, len(fact) + 1))

dims = {}
for col, key in [
    ("gender", "gender_key"),
    ("race_ethnicity", "ethnicity_key"),
    ("parental_education", "parental_education_key"),
    ("lunch", "lunch_key"),
    ("test_prep", "test_prep_key"),
]:
    dim = pd.DataFrame({col: sorted(fact[col].unique())})
    dim[key] = np.arange(1, len(dim) + 1)
    dims[col] = dim[[key, col]]

for col, key in [
    ("gender", "gender_key"),
    ("race_ethnicity", "ethnicity_key"),
    ("parental_education", "parental_education_key"),
    ("lunch", "lunch_key"),
    ("test_prep", "test_prep_key"),
]:
    mapper = dict(zip(dims[col][col], dims[col][key]))
    fact[key] = fact[col].map(mapper)

fact_preview = fact[
    ["student_id", "gender_key", "ethnicity_key", "parental_education_key",
     "lunch_key", "test_prep_key"] + score_cols
].head()

display(fact_preview)
for name, dim in dims.items():
    print(f"Dim_{name}")
    display(dim)


## 6. Executive Dashboard

In [ ]:
# Dashboard-style KPI cards
fig = plt.figure(figsize=(16, 9))
fig.suptitle("Student Performance — Executive Dashboard", fontsize=20, fontweight="bold", y=0.98)

cards = [
    ("Students", f"{len(df):,}"),
    ("Math Avg", f"{df.math_score.mean():.1f}"),
    ("Reading Avg", f"{df.reading_score.mean():.1f}"),
    ("Writing Avg", f"{df.writing_score.mean():.1f}"),
    ("All-Subject Pass", f"{overall_pass_rate:.1f}%"),
]

for i, (label, value) in enumerate(cards):
    ax = fig.add_axes([0.03 + i*0.19, 0.76, 0.17, 0.14])
    ax.axis("off")
    ax.text(0.5, 0.65, value, ha="center", va="center", fontsize=22, fontweight="bold")
    ax.text(0.5, 0.18, label, ha="center", va="center", fontsize=10)

# Main chart
ax1 = fig.add_axes([0.06, 0.39, 0.42, 0.30])
tmp = df.groupby("test_prep")[score_cols].mean()
tmp.plot(kind="bar", ax=ax1)
ax1.set_title("Scores by Test Preparation")
ax1.set_ylabel("Average score")
ax1.set_ylim(0, 100)
ax1.tick_params(axis="x", rotation=0)

# Gap chart
ax2 = fig.add_axes([0.55, 0.39, 0.39, 0.30])
gap_chart = pd.Series({
    "Test prep": prep_gap.loc["overall_score", "gap_points"],
    "Lunch": lunch_gap.loc["overall_score", "gap_points"],
    "Largest education gap": largest_pair["gap_points"],
})
gap_chart.sort_values().plot(kind="barh", ax=ax2)
ax2.set_title("Largest Segment Gaps (points)")
ax2.set_xlabel("Point difference")
ax2.axvline(0, linewidth=1)

# Correlation chart
ax3 = fig.add_axes([0.06, 0.07, 0.42, 0.23])
sns.heatmap(corr, annot=True, fmt=".3f", cmap="Blues", ax=ax3, cbar=False)
ax3.set_title("Subject Correlations")

# Why callout
ax4 = fig.add_axes([0.55, 0.07, 0.39, 0.23])
ax4.axis("off")
ax4.text(0, 0.95, "WHY THIS GAP?", fontsize=13, fontweight="bold", va="top")
ax4.text(
    0, 0.72,
    f"Largest detected gap: {g1} vs {g2} parental education "
    f"= {largest_pair['gap_points']:.1f} overall points.",
    fontsize=11, va="top"
)
ax4.text(
    0, 0.48,
    f"Writing contributes the largest subject gap "
    f"({subject_gap.loc['writing_score', 'gap_points']:.1f} points).",
    fontsize=11, va="top"
)
ax4.text(
    0, 0.24,
    "This is an observed association, not a causal finding.",
    fontsize=10, va="top"
)

plt.show()


## 7. Executive Notebook — Presentation Summary

### Highlights
- 1,000 students are represented.
- Average performance is strongest in reading and lowest in math.
- Reading and writing are extremely strongly correlated.
- Test-prep completion is associated with higher performance across all subjects.
- Segment gaps are visible by lunch type and parental education.

### Key KPIs
- Math average: **66.1**
- Reading average: **69.2**
- Writing average: **68.1**
- All-subject pass rate at ≥60: **60.3%**
- Overall score standard deviation: reported in the KPI table above.

### Major Insights
1. **Test preparation:** completed students outperform students with no preparation, with writing showing the largest subject gap.
2. **Gender:** the direction of the gap differs by subject; it should not be summarized as a single universal advantage.
3. **Parental education:** average scores vary across education categories and show a weak positive ordered relationship.
4. **Lunch:** the standard vs. free/reduced gap remains visible within parental-education groups.

### Root Causes / Evidence
The analysis does not claim causal root causes. Instead, it performs stratified breakdowns to determine whether observed gaps persist across other available variables.

### Problems / Risks
- Persistent segment gaps may represent equity and performance-monitoring risks.
- The dataset does not identify schools, locations, socioeconomic measures beyond lunch, or student histories.
- Categorical fields have no external validation source.

### Opportunities
- Evaluate targeted access to test-preparation resources.
- Monitor writing performance because it contributes strongly to several observed gaps.
- Use stratified dashboards rather than relying only on overall averages.

### Recommendations
1. Prioritize further evaluation of test-prep access and outcomes.
2. Investigate persistent lunch-related gaps with richer socioeconomic/contextual data.
3. Monitor subject-level differences instead of using only one composite score.
4. In Power BI, implement the recommended star schema and interactive segment filters.

### Next Steps
- Add school/contextual variables if available.
- Validate category definitions with a trusted source.
- If longitudinal data becomes available, add cohort and time intelligence.
- Test interventions with stronger causal designs before labeling any factor as a cause.


## 8. Portfolio / CV Summary

- **Built a Python-based educational BI analysis of 1,000 student records**, performing data-quality validation, KPI development, segment analysis, correlation analysis, and statistical testing to identify performance and equity gaps.
- **Quantified student-performance differences across test preparation, lunch, gender, and parental education segments**, including a +7.6-point overall association for completed test preparation and subject-level gap diagnostics.
- **Designed a Power BI/Excel-ready star-schema model and executive dashboard framework**, translating descriptive and statistical findings into actionable educational monitoring recommendations while explicitly avoiding unsupported causal claims.


## 9. Limitations

1. **No time dimension:** MoM/QoQ/YoY trends cannot be calculated.
2. **No school/geography field:** institutional or geographic comparisons are impossible.
3. **Single-source sample:** the dataset is relatively small and may not generalize beyond its source population.
4. **Categorical fields lack validation:** labels are accepted as supplied and are not externally verified.
5. **Observational data:** statistically significant associations do not prove causation.
6. **No student identifier:** a surrogate `student_id` is generated solely for modeling convenience.
7. **No intervention data:** the analysis cannot establish whether test preparation itself caused the observed score differences.


---

**Reproducibility note:** Put `StudentsPerformance.csv` in the same folder as this notebook and run all cells from top to bottom. The notebook preserves the original source data and creates derived analysis objects in memory only.


# ✅ Final Portfolio Checklist

Before publishing this project:

- [ ] `StudentsPerformance.csv` is included or its source is documented.
- [ ] Notebook runs from top to bottom without errors.
- [ ] Data quality results are visible.
- [ ] KPI definitions are stated.
- [ ] Every major gap has actual numbers behind it.
- [ ] Statistical significance is reported where appropriate.
- [ ] Causation is not claimed from observational data.
- [ ] No time trend is fabricated.
- [ ] Dashboard is readable.
- [ ] Star-schema recommendation is documented.
- [ ] Limitations are explicit.
- [ ] CV bullets are quantified and ATS-friendly.
